# 05. HAWQ: Mixed-Precision Bit Allocation and Numeric Debugging

- **Category:** 14_quantisation / 05_kv_attention_optim_and_benchmarks
- **Lineage & Citations:** Dong et al. (2019, *HAWQ: Hessian AWare Quantization of Neural Networks with Mixed-Precision*, ICCV); Yao et al. (2021, *HAWQ-V3: Dyadic Neural Network Quantization*, ICML).
- **Purpose:** Formulate optimal non-uniform bit allocation across neural network layers using second-order Hessian curvature trace estimation $\text{Tr}(H_l) = \mathbb{E}[z^\top H_l z]$ and greedy Pareto Integer Linear Programming (ILP).
- **Topic Connections:** `🔗 Upstream Prerequisites:` [04. Quantized Gradients QSGD TernGrad and 1bit Adam Error Feedback](04_quantized_gradients_qsgd_terngrad_and_1bit_adam_error_feedback.ipynb) $\to$ `🔗 Downstream Syntheses:` [06. Quantization Method Selection Matrix and Head to Head Benchmark](06_quantization_method_selection_matrix_and_head_to_head_benchmark.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do NOT apply uniform low-bit quantization when layer Hessian traces differ by more than $10\times$ (underallocating bits to high-curvature layers causes irreversible SQNR collapse); do NOT compute explicit Hessians for large matrices ($H \in \mathbb{R}^{d \times d}$ exceeds memory capacity; always use Hutchinson matrix-free probe estimation).

## Mathematical Foundations & Governing Equations

1. **Second-Order Loss Perturbation Expansion**:
   $$\Delta \mathcal{L} \approx \frac{1}{2} \sum_{l=1}^L \Delta W_l^\top H_l \Delta W_l \approx \frac{1}{2} \sum_{l=1}^L \bar{\text{Tr}}(H_l) \cdot \|W_l - Q_{b_l}(W_l)\|_F^2$$
2. **Hutchinson Randomized Trace Estimator**:
   $$\text{Tr}(H_l) = \mathbb{E}_{z \sim \text{Rademacher}} [z^\top H_l z] \approx \frac{1}{M} \sum_{m=1}^M z_m^\top (H_l z_m)$$
3. **Greedy Pareto Bit Allocation Marginal Efficiency**:
   $$\text{Gain}(l) = \frac{\Omega_l(b_l) - \Omega_l(b_l')}{(b_l' - b_l) \cdot d_l}, \quad \Omega_l(b) = \frac{1}{2} \bar{\text{Tr}}(H_l) \cdot \|W_l - Q_b(W_l)\|_F^2$$
4. **Layerwise Signal-to-Quantization-Noise Ratio (SQNR)**:
   $$\text{SQNR}_l = 10 \log_{10}\left( \frac{\frac{1}{B}\sum a_{l}^2}{\frac{1}{B}\sum (a_l - \hat{a}_l)^2 + \epsilon} \right)$$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

torch.manual_seed(42)
r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
WEIGHTS_PATH = r / "weights/14_quantisation/05_kv_attention_optim_and_benchmarks/05_hawq_mixed_precision_bit_allocation_and_numeric_debugging.safetensors"
weights = load_file(str(WEIGHTS_PATH))
print(f"Loaded {len(weights)} tensors from {WEIGHTS_PATH.name}")

In [ ]:
def quantize_weight_bits(w: torch.Tensor, bits: int) -> torch.Tensor:
    levels = float((2 ** (bits - 1)) - 1)
    scale = torch.amax(torch.abs(w), dim=-1, keepdim=True) / levels
    q = torch.clamp(torch.round(w / (scale + 1e-8)), -levels, levels)
    return q * scale

def compute_layer_sensitivity(w: torch.Tensor, trace: float, candidate_bits: list[int] = [2, 4, 8]) -> dict[int, float]:
    sens = {}
    for b in candidate_bits:
        w_q = quantize_weight_bits(w, b)
        sens[b] = 0.5 * trace * float(torch.sum((w - w_q) ** 2))
    return sens

def greedy_hawq_bit_allocation(layer_weights, layer_traces, target_avg_bits=4.0, candidate_bits=[2, 4, 8]):
    numels = [w.numel() for w in layer_weights]
    max_total_bits = target_avg_bits * sum(numels)
    sens = [compute_layer_sensitivity(w, tr, candidate_bits) for w, tr in zip(layer_weights, layer_traces)]
    alloc = [min(candidate_bits) for _ in layer_weights]

    while True:
        best_layer, best_ratio, best_next_b = None, -1.0, None
        for l in range(len(alloc)):
            cur_b = alloc[l]
            idx = candidate_bits.index(cur_b)
            if idx + 1 < len(candidate_bits):
                next_b = candidate_bits[idx + 1]
                cost = (next_b - cur_b) * numels[l]
                if sum(b * n for b, n in zip(alloc, numels)) + cost <= max_total_bits:
                    ratio = (sens[l][cur_b] - sens[l][next_b]) / cost
                    if ratio > best_ratio:
                        best_ratio, best_layer, best_next_b = ratio, l, next_b
        if best_layer is None: break
        alloc[best_layer] = best_next_b
    return alloc

def layerwise_sqnr_diagnostic(x, weights, bit_allocation):
    a_fp, a_q, sqnrs = x, x, []
    for w, b in zip(weights, bit_allocation):
        w_hat = quantize_weight_bits(w, b)
        a_fp = a_fp @ w
        a_q = a_q @ w_hat
        sqnr = 10.0 * torch.log10(torch.mean(a_fp ** 2) / (torch.mean((a_fp - a_q) ** 2) + 1e-12))
        sqnrs.append(sqnr)
    return torch.tensor(sqnrs)

In [ ]:
# Parity Verification against golden safetensors checkpoint
y_anchor = weights["test_input"] @ weights["weight"] + weights["bias"]
assert float(torch.max(torch.abs(y_anchor - weights["test_output"]))) < 1e-5

w_list = [weights["w0"], weights["w1"], weights["w2"], weights["w3"]]
tr_list = weights["traces"].tolist()

alloc_3_5 = greedy_hawq_bit_allocation(w_list, tr_list, target_avg_bits=3.5)
assert torch.allclose(torch.tensor(alloc_3_5, dtype=torch.float32), weights["alloc_target_3_5"])

alloc_5_0 = greedy_hawq_bit_allocation(w_list, tr_list, target_avg_bits=5.0)
assert torch.allclose(torch.tensor(alloc_5_0, dtype=torch.float32), weights["alloc_target_5_0"])

sqnrs = layerwise_sqnr_diagnostic(weights["x"], w_list, alloc_5_0)
assert float(torch.max(torch.abs(sqnrs - weights["sqnrs"]))) < 1e-4
print("All HAWQ bit allocation and layerwise SQNR assertions verified successfully!")

In [ ]:
# Empirical Pareto Comparison: Uniform 2-bit vs Uniform 4-bit vs HAWQ [8, 4, 4, 4]
x_eval = weights["x"]
sqnr_u2 = layerwise_sqnr_diagnostic(x_eval, w_list, [2, 2, 2, 2])[-1].item()
sqnr_u4 = layerwise_sqnr_diagnostic(x_eval, w_list, [4, 4, 4, 4])[-1].item()
sqnr_hawq = layerwise_sqnr_diagnostic(x_eval, w_list, alloc_5_0)[-1].item()

print(f"Uniform 2-bit Final SQNR: {sqnr_u2:6.2f} dB (Catastrophic collapse)")
print(f"Uniform 4-bit Final SQNR: {sqnr_u4:6.2f} dB (Baseline)")
print(f"HAWQ [8,4,4,4] Final SQNR: {sqnr_hawq:6.2f} dB (+{sqnr_hawq - sqnr_u4:.2f} dB gain over 4-bit)")
assert sqnr_hawq > sqnr_u4, "HAWQ mixed precision must outperform uniform 4-bit!"

In [ ]:
# Latency Microbenchmark: Greedy Bit Allocation across 32 transformer layers
dummy_weights = [torch.randn(64, 64) for _ in range(32)]
dummy_traces = [float(torch.rand(1).item() * 10.0 + 0.5) for _ in range(32)]

t0 = time.perf_counter()
for _ in range(20):
    _ = greedy_hawq_bit_allocation(dummy_weights, dummy_traces, target_avg_bits=4.0)
t_alloc = (time.perf_counter() - t0) / 20 * 1e3

print(f"HAWQ Greedy Pareto ILP Allocation (32 layers) Latency: {t_alloc:.3f} ms")
assert t_alloc < 50.0, "Bit allocation optimization must complete in <50 ms"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Curvature Governs Sensitivity**: Layers with high Hessian traces must receive higher bitwidths; allocating 8 bits to the top $10\%$ of high-curvature layers avoids severe activation degradation.
2. **Matrix-Free Hutchinson Probes**: Using random Rademacher vector projections allows accurate layerwise trace estimation in $\mathcal{O}(M \cdot d)$ time without storing the $d \times d$ explicit Hessian.
3. **Diagnostic SQNR Shadow Profiling**: Running dual FP32 and quantized forward passes pinpoint the exact layer where numerical instability or outlier activation clipping occurs.

---
### Synchronized Track Navigation
- [Companion Guide](../../../markdown/14_quantisation/05_kv_attention_optim_and_benchmarks/05_hawq_mixed_precision_bit_allocation_and_numeric_debugging.md)
- [PyTorch Reference](../../../pytorch2/14_quantisation/05_kv_attention_optim_and_benchmarks/05_hawq_mixed_precision_bit_allocation_and_numeric_debugging.py)
- [JAX Reference](../../../jax/14_quantisation/05_kv_attention_optim_and_benchmarks/05_hawq_mixed_precision_bit_allocation_and_numeric_debugging.py)
- [Candle Reference](../../../candle/14_quantisation/05_kv_attention_optim_and_benchmarks/05_hawq_mixed_precision_bit_allocation_and_numeric_debugging.rs)
- [Kernels Reference](../../../kernels/14_quantisation/05_kv_attention_optim_and_benchmarks/05_hawq_mixed_precision_bit_allocation_and_numeric_debugging.rs)
- `🔗 Upstream Prerequisites:` [04. Quantized Gradients QSGD TernGrad and 1bit Adam Error Feedback](04_quantized_gradients_qsgd_terngrad_and_1bit_adam_error_feedback.ipynb)
- `🔗 Downstream Syntheses:` [06. Quantization Method Selection Matrix and Head to Head Benchmark](06_quantization_method_selection_matrix_and_head_to_head_benchmark.ipynb)